In [ ]:

def plot_city_relative_trends(tidy_data, cities_to_plot=None, max_plots=160):
    """
    Plot relative emissions and wellbeing trends for each city with proper sizing and legends
    """
    # Get unique cities
    if cities_to_plot is None:
        all_cities = tidy_data['Municipality_Name'].unique()
        cities_to_plot = all_cities[:max_plots]
    
    # Define metrics to plot
    emissions_metrics = [
        'Utility_Total_Emissions', 'Transportation_Emissions',
        'Utility_Res_Emissions', 'Utility_CSMI_Emissions', 
        'Utility_ELEC', 'Utility_Res_NG', 'Utility_Res_RNG','Utility_CSMI_NG', 'Utility_CSMI_RNG', 'Utility_Wood', 
        'Utility_Emission_Oil', 'Utility_DPRO',
    ]
    
    wellbeing_metrics = ['CWB', 'Income', 'Education', 'Housing', 'Labour_Force', 'Population']
    
    # Create individual plots for each city (larger and clearer)
    for city in cities_to_plot:
        fig, ax = plt.subplots(figsize=(14, 8))
        city_data = tidy_data[tidy_data['Municipality_Name'] == city]
        
        # Track if we plotted anything
        plotted_emissions = []
        plotted_wellbeing = []
        
        # Plot emissions (relative to 2007)
        for metric in emissions_metrics:
            metric_data = city_data[city_data['Metric'] == metric].sort_values('Year')
            if len(metric_data) > 0:
                baseline_2007 = metric_data[metric_data['Year'] == 2007]['Value'].values
                if len(baseline_2007) > 0 and baseline_2007[0] > 0:
                    relative_values = metric_data.set_index('Year')['Value'] / baseline_2007[0]
                    label = metric.replace('Utility_Emissions_', '').replace('Utility_', '').replace('_', ' ')
                    ax.plot(relative_values.index, relative_values.values, 
                           linestyle='-', label=f'{label} (emissions)',
                           alpha=1, linewidth=2)
                    plotted_emissions.append(metric)
        
        # Plot wellbeing metrics (relative to 2006)
        for metric in wellbeing_metrics:
            metric_data = city_data[city_data['Metric'] == metric].sort_values('Year')
            if len(metric_data) > 0:
                baseline_2006 = metric_data[metric_data['Year'] == 2006]['Value'].values
                if len(baseline_2006) > 0 and baseline_2006[0] > 0:
                    relative_values = metric_data.set_index('Year')['Value'] / baseline_2006[0]
                    ax.plot(relative_values.index, relative_values.values,
                           linestyle='--', label=f'{metric} (wellbeing)', 
                           alpha=0.8, linewidth=2.5, marker='o', markersize=4)
                    plotted_wellbeing.append(metric)
        
        # Formatting
        ax.axhline(y=1.0, color='black', linestyle='-', linewidth=2, alpha=1)
        ax.set_title(f'{city}', fontsize=16, fontweight='bold')
        ax.set_xlabel('Year', fontsize=12)
        ax.set_ylabel('Relative Value (Base Year = 1.0)', fontsize=12)
        ax.grid(True, alpha=0.3)
        ax.set_ylim(0, 3)
        
        # Create legend with two columns
        if plotted_emissions or plotted_wellbeing:
            ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1), 
                     ncol=1, fontsize=10, frameon=True, fancybox=True)
        
        plt.tight_layout()
        plt.show()


# Run the plotting
plot_city_relative_trends(tidy_all_data)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.patches import Patch

def plot_energy_transitions(df, municipality, plot_type='utility'):
    """
    Create a stacked area chart showing energy transition for a specific municipality
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe containing the data
    municipality : str
        Name of the municipality to plot
    plot_type : str
        Either 'utility' or 'transportation'
    """
    
    # Filter data for the specific municipality
    muni_data = df[df['Municipality_Name'] == municipality].copy()
    
    # Define metrics based on plot type
    if plot_type == 'residential_utility':
        metrics = ['Utility_Residential_Electricity_Share', 'Utility_Residential_Polluting_Share']
        title_suffix = 'Utility Residential Energy'
        colors = ['#66c2a5', '#fc8d62']  # Green for electricity, orange for polluting
        labels = ['Electricity', 'Polluting Sources']
    elif plot_type == 'csmi_utility':
        metrics = ['Utility_CSMI_Electricity_Share', 'Utility_CSMI_Polluting_Share']
        title_suffix = 'Utility CSMI Energy'
        colors = ['#66c2a5', '#fc8d62']  # Green for electricity, orange for polluting
        labels = ['Electricity', 'Polluting Sources']
    elif plot_type == 'transportation':
        metrics = ['Low_Emissions_km_Share', 'Polluting_km_Share']
        title_suffix = 'Transportation'
        colors = ['#8dd3c7', '#fb8072']  # Light green for low emissions, red for polluting
        labels = ['Low Emissions', 'Polluting Vehicles']
    
    # Filter for relevant metrics
    plot_data = muni_data[muni_data['Metric'].isin(metrics)]
    
    if plot_data.empty:
        print(f"No {plot_type} data found for {municipality}")
        return None
    
    # Pivot the data to have years as index and metrics as columns
    pivot_data = plot_data.pivot(index='Year', columns='Metric', values='Value')
    
    # Ensure all metrics are present (fill with 0 if missing)
    for metric in metrics:
        if metric not in pivot_data.columns:
            pivot_data[metric] = 0
    
    # Sort by year
    pivot_data = pivot_data.sort_index()
    
    # Normalize to ensure sum is 100% (in case data doesn't already sum to 100)
    row_sums = pivot_data.sum(axis=1)
    pivot_data = pivot_data.div(row_sums, axis=0) * 100
    
    # Create the figure
    fig, ax = plt.subplots(figsize=(14, 8))
    
    # Create x values (years)
    x = pivot_data.index.values
    
    # Stack the areas
    y_stack = []
    for i, metric in enumerate(metrics):
        if metric in pivot_data.columns:
            y_stack.append(pivot_data[metric].values)
        else:
            y_stack.append(np.zeros(len(x)))
    
    # Create the stacked area plot
    ax.stackplot(x, *y_stack, labels=labels, colors=colors, alpha=0.8)
    
    # Styling to match the reference image
    ax.set_ylim(0, 100)
    ax.set_xlim(x.min(), x.max())
    
    # Add percentage labels on y-axis
    ax.set_ylabel('Share of energy (%)', fontsize=12)
    ax.set_yticks([0, 20, 40, 60, 80, 100])
    ax.set_yticklabels(['0%', '20%', '40%', '60%', '80%', '100%'])
    
    # Grid styling
    ax.grid(True, alpha=0.3, linestyle='-', linewidth=0.5)
    ax.set_axisbelow(True)
    
    # Title
    ax.set_title(f'Long-term energy transitions, {municipality}\n{title_suffix}', 
                 fontsize=16, fontweight='bold', pad=20)
    
    # Subtitle
    subtitle_text = (f'Share of {plot_type} energy by source over time, '
                    f'measured as the percentage of total consumption.')
    ax.text(0.5, 1.02, subtitle_text, transform=ax.transAxes,
            ha='center', fontsize=11, color='#666666')
    
    # Legend
    ax.legend(loc='center left', bbox_to_anchor=(1.02, 0.5), 
             frameon=False, fontsize=11)
    
    # Remove top and right spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Add source information
    source_text = 'Data source: BC Municipal Energy Data'
    fig.text(0.1, 0.02, source_text, fontsize=9, color='#666666')
    
    plt.tight_layout()
    
    return fig

def plot_all_municipalities(df, plot_type='utility', save_plots=False, output_dir='plots'):
    """
    Create energy transition plots for all municipalities in the dataframe
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe containing the data
    plot_type : str
        Either 'utility' or 'transportation'
    save_plots : bool
        Whether to save the plots to files
    output_dir : str
        Directory to save plots if save_plots is True
    """
    
    # Get unique municipalities
    municipalities = df['Municipality_Name'].unique()
    
    # Create output directory if saving plots
    if save_plots:
        import os
        os.makedirs(output_dir, exist_ok=True)
    
    figures = {}
    
    for municipality in municipalities:
        print(f"Creating {plot_type} plot for {municipality}...")
        fig = plot_energy_transitions(df, municipality, plot_type)
        
        if fig is not None:
            figures[municipality] = fig
            
            if save_plots:
                filename = f"{output_dir}/{municipality}_{plot_type}_transition.png"
                fig.savefig(filename, dpi=300, bbox_inches='tight')
                plt.close(fig)
    
    return figures

# Main execution
if __name__ == "__main__":
    # Load your actual data here
    # df = pd.read_csv('your_data.csv')
    
    # For demonstration, create sample data
    df = tidy_all_data.copy()
    
    # Create utility plots for all municipalities
    print("\nCreating Utility Residential Energy Transition Plots...")
    utility_figures = plot_all_municipalities(df, plot_type='residential_utility', save_plots=False)
    
    print("\nCreating Utility CSMI Energy Transition Plots...")
    utility_figures = plot_all_municipalities(df, plot_type='csmi_utility', save_plots=False)
    
    # Create transportation plots for all municipalities
    print("\nCreating Transportation Energy Transition Plots...")
    transport_figures = plot_all_municipalities(df, plot_type='transportation', save_plots=False)
    
    # Display the plots (if not saving to files)
    if utility_figures or transport_figures:
        plt.show()
    
    print("\nPlots created successfully!")


In [ ]:
# Additional function to create a single combined plot with subplots
def plot_municipality_overview(df, municipality):
    """
    Create a combined plot showing both utility and transportation transitions
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The dataframe containing the data
    municipality : str
        Name of the municipality to plot
    """
    
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 7))
    
    # Utility plot
    muni_data = df[df['Municipality_Name'] == municipality].copy()
    
    # Process utility data for residences
    utility_residential_metrics = ['Transportation_Low_Emissions_Vehicle_km_Travelled_Share', 'Transportation_Polluting_Vehicle_km_Travelled_Share']
    utility_residential_data = muni_data[muni_data['Metric'].isin(utility_residential_metrics)]
    
    if not utility_residential_data.empty:
        pivot_utility = utility_residential_data.pivot(index='Year', columns='Metric', values='Value')
        row_sums = pivot_utility.sum(axis=1)
        pivot_utility = pivot_utility.div(row_sums, axis=0) * 100
        
        x = pivot_utility.index.values
        y_utility = [pivot_utility[m].values if m in pivot_utility.columns else np.zeros(len(x)) 
                     for m in utility_residential_metrics]
        
        ax1.stackplot(x, *y_utility, 
                     labels=['Electricity', 'Polluting Sources'],
                     colors=['#66c2a5', '#fc8d62'], alpha=0.8)
        ax1.set_title(f'Utility Residential Energy Transition', fontsize=14)
        ax1.set_ylabel('Share (%)')
        ax1.set_ylim(0, 100)
        ax1.legend(loc='upper left', fontsize=10)
        ax1.grid(True, alpha=0.3)
    
    # Process utility data for residences
    utility_CSMI_metrics = ['Utility_CSMI_Electricity_Share', 'Utility_CSMI_Polluting_Share']
    utility_CSMI_data = muni_data[muni_data['Metric'].isin(utility_CSMI_metrics)]
    
    if not utility_CSMI_data.empty:
        pivot_utility = utility_CSMI_data.pivot(index='Year', columns='Metric', values='Value')
        row_sums = pivot_utility.sum(axis=1)
        pivot_utility = pivot_utility.div(row_sums, axis=0) * 100
        
        x = pivot_utility.index.values
        y_utility = [pivot_utility[m].values if m in pivot_utility.columns else np.zeros(len(x)) 
                     for m in utility_CSMI_metrics]
        
        ax2.stackplot(x, *y_utility, 
                     labels=['Electricity', 'Polluting Sources'],
                     colors=['#66c2a5', '#fc8d62'], alpha=0.8)
        ax2.set_title(f'Utility CSMI Energy Transition', fontsize=14)
        ax2.set_ylabel('Share (%)')
        ax2.set_ylim(0, 100)
        ax2.legend(loc='upper left', fontsize=10)
        ax2.grid(True, alpha=0.3)
    
    
    # Transportation plot
    transport_metrics = ['Low_Emissions_km_Share', 'Polluting_km_Share']
    transport_data = muni_data[muni_data['Metric'].isin(transport_metrics)]
    
    if not transport_data.empty:
        pivot_transport = transport_data.pivot(index='Year', columns='Metric', values='Value')
        row_sums = pivot_transport.sum(axis=1)
        pivot_transport = pivot_transport.div(row_sums, axis=0) * 100
        
        x = pivot_transport.index.values
        y_transport = [pivot_transport[m].values if m in pivot_transport.columns else np.zeros(len(x))
                       for m in transport_metrics]
        
        ax3.stackplot(x, *y_transport,
                     labels=['Low Emissions', 'Polluting Vehicles'],
                     colors=["#4ee2c9", "#fe5b48"], alpha=0.8)
        ax3.set_title(f'Transportation Energy Transition', fontsize=14)
        ax3.set_ylabel('Share (%)')
        ax3.set_ylim(0, 100)
        ax3.legend(loc='upper left', fontsize=10)
        ax3.grid(True, alpha=0.3)
    
    fig.suptitle(f'Energy Transitions Overview - {municipality}', 
                 fontsize=16, fontweight='bold', y=1.02)
    
    plt.tight_layout()
    return fig

In [ ]:
plot_municipality_overview(df=tidy_all_data, municipality='Burnaby')